# Microbial Diversity Analysis

Microorganisms represent the vast majority of biological diversity on Earth. Studying microbial communities -- who is there, how many, and how they differ across environments -- is central to fields from ecology to clinical medicine. This notebook covers the computational tools and concepts used to analyze microbial diversity from amplicon sequencing data.

---

## Learning Objectives

By the end of this notebook, you will be able to:

1. Explain the 16S rRNA gene approach to microbial community profiling
2. Distinguish between OTU clustering and ASV denoising
3. Compute and interpret alpha diversity metrics (observed species, Shannon, Simpson, Faith's PD)
4. Construct rarefaction curves and explain their purpose
5. Compute and interpret beta diversity metrics (Bray-Curtis, UniFrac)
6. Apply ordination (PCoA, NMDS) to visualize community differences
7. Perform statistical testing (PERMANOVA, ANOSIM) on distance matrices
8. Create standard microbiome visualizations (stacked bar plots, diversity boxplots, ordination plots)

---

[← Previous: RNA-seq Analysis: From Reads to Differential Expression](../../Tier_3_Applied_Bioinformatics/03_RNA_seq_Analysis/01_rna_seq_analysis.ipynb) | [Next: Promoter and Regulatory Sequence Analysis →](../../Tier_3_Applied_Bioinformatics/05_Promoter_and_Regulatory_Analysis/01_promoter_and_regulatory_analysis.ipynb)

---

## 1. The 16S rRNA Gene and Amplicon Sequencing

### Why 16S rRNA?

The 16S ribosomal RNA gene is the standard marker for bacterial identification because it:

- Is present in **all bacteria and archaea** (universal marker)
- Contains **conserved regions** (good primer binding sites) alternating with **variable regions** (V1-V9, which differ between species)
- Has a **large reference database** (SILVA, Greengenes, RDP)
- Is ~1500 bp long -- suitable for sequencing

```
16S rRNA gene (~1500 bp):

  V1   V2   V3   V4   V5   V6   V7   V8   V9
|----|----|----|----|----|----|----|----|----|----|
  C    C    C    C    C    C    C    C    C    C

C = conserved region    V = variable region
```

### Common Primer Targets

| Region | Primers | Length | Notes |
|--------|---------|--------|-------|
| V4 | 515F/806R | ~250 bp | Earth Microbiome Project standard |
| V3-V4 | 341F/785R | ~460 bp | Good taxonomic resolution |
| V1-V3 | 27F/534R | ~500 bp | Alternative target |

### Other Marker Genes

| Marker | Target | Use |
|--------|--------|-----|
| 18S rRNA | Eukaryotes | Fungi, protists |
| ITS (Internal Transcribed Spacer) | Fungi | Better fungal resolution than 18S |
| Shotgun metagenomics | All DNA | Full functional + taxonomic profiling |

### Amplicon Sequencing Workflow

```
Environmental sample (soil, gut, water, etc.)
      |
      v
DNA extraction
      |
      v
PCR amplification of 16S rRNA gene (target V region)
      |
      v
Library preparation (add barcodes + adapters)
      |
      v
Sequencing (Illumina MiSeq: paired-end 250-300 bp)
      |
      v
Bioinformatics pipeline:
  - Quality filtering
  - Denoising (ASVs) or clustering (OTUs)
  - Taxonomy assignment
  - Diversity analysis
```

## 2. OTU Clustering vs. ASV Denoising

After quality filtering, we need to group sequences into meaningful biological units.

### OTU Clustering (Traditional)

**Operational Taxonomic Units (OTUs)** group sequences by similarity (typically 97%, roughly corresponding to species).

Clustering approaches:
- **De novo**: Cluster sequences against each other (e.g., UCLUST, VSEARCH)
- **Closed reference**: Map sequences to a reference database
- **Open reference**: Closed reference first, then de novo on unmatched reads

The 97% threshold is a convention. The actual relationship between sequence similarity and species boundaries varies across taxonomic groups.

### ASV Denoising (Modern)

**Amplicon Sequence Variants (ASVs)** use statistical models to distinguish true biological sequences from sequencing errors, resolving sequences down to single-nucleotide differences.

| Feature | OTUs (97%) | ASVs |
|---------|-----------|------|
| Resolution | Species-level (approximate) | Single-nucleotide |
| Reproducibility | Depends on clustering algorithm | Consistent across studies |
| Biological meaning | Arbitrary threshold | Exact sequences |
| Database dependency | Some methods need reference | Reference-free |
| Tools | UCLUST, VSEARCH, CD-HIT | DADA2, Deblur, UNOISE3 |

**ASVs are now the recommended approach** (Callahan et al., 2017).

### DADA2 Algorithm (Simplified)

DADA2 builds an error model from the data:
1. Learn the error rates for each possible nucleotide transition at each quality score
2. For each unique sequence, test whether it could be explained as an error from a more abundant sequence
3. Output: a set of corrected (denoised) sequences with exact counts

## 3. Taxonomy Assignment

After obtaining OTUs or ASVs, we assign taxonomy to each sequence.

### Methods

| Method | Approach | Example |
|--------|----------|--------|
| Naive Bayes classifier | Machine learning on k-mer frequencies | QIIME2 classify-sklearn |
| BLAST/VSEARCH | Sequence similarity search | Top hit or consensus |
| Phylogenetic placement | Place on reference tree | pplacer, EPA-ng |

### Taxonomy Ranks

```
Kingdom: Bacteria
  Phylum: Firmicutes
    Class: Bacilli
      Order: Lactobacillales
        Family: Lactobacillaceae
          Genus: Lactobacillus
            Species: Lactobacillus acidophilus
```

### Reference Databases

| Database | Size | Notes |
|----------|------|-------|
| SILVA | ~2M sequences | Most comprehensive, regularly updated |
| Greengenes2 | Standardized taxonomy | Uses genome phylogeny |
| RDP | ~3M sequences | Long-standing, Naive Bayes classifier |
| GTDB | Genome-based | Gold standard for genome taxonomy |

## 4. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from scipy import stats
from scipy.spatial.distance import pdist, squareform
from scipy.cluster.hierarchy import linkage, dendrogram
from sklearn.manifold import MDS
from itertools import combinations

np.random.seed(42)

plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 11

## 5. The Feature Table

The central data structure in microbial diversity analysis is the **feature table** (also called an OTU table or BIOM table):

- Rows = features (OTUs or ASVs)
- Columns = samples
- Values = counts (number of reads assigned to each feature in each sample)

Let's create a simulated dataset representing a human microbiome study with samples from three body sites.

In [ ]:
def simulate_microbiome_data(n_samples_per_site=8, n_taxa=50, total_reads_range=(5000, 20000)):
    """
    Simulate a microbiome feature table with samples from three body sites.
    Each body site has a characteristic community composition.
    """
    sites = ['Gut', 'Skin', 'Oral']
    n_samples = n_samples_per_site * len(sites)
    
    # Define taxa with phylum-level classification
    taxa_info = {
        'Bacteroides_sp1': 'Bacteroidetes', 'Bacteroides_sp2': 'Bacteroidetes',
        'Prevotella_sp1': 'Bacteroidetes', 'Prevotella_sp2': 'Bacteroidetes',
        'Alistipes_sp1': 'Bacteroidetes',
        'Faecalibacterium_sp1': 'Firmicutes', 'Roseburia_sp1': 'Firmicutes',
        'Ruminococcus_sp1': 'Firmicutes', 'Clostridium_sp1': 'Firmicutes',
        'Clostridium_sp2': 'Firmicutes', 'Lactobacillus_sp1': 'Firmicutes',
        'Lactobacillus_sp2': 'Firmicutes', 'Streptococcus_sp1': 'Firmicutes',
        'Streptococcus_sp2': 'Firmicutes', 'Streptococcus_sp3': 'Firmicutes',
        'Staphylococcus_sp1': 'Firmicutes', 'Staphylococcus_sp2': 'Firmicutes',
        'Enterococcus_sp1': 'Firmicutes', 'Veillonella_sp1': 'Firmicutes',
        'Dialister_sp1': 'Firmicutes',
        'Bifidobacterium_sp1': 'Actinobacteria', 'Bifidobacterium_sp2': 'Actinobacteria',
        'Corynebacterium_sp1': 'Actinobacteria', 'Corynebacterium_sp2': 'Actinobacteria',
        'Propionibacterium_sp1': 'Actinobacteria', 'Cutibacterium_sp1': 'Actinobacteria',
        'Escherichia_sp1': 'Proteobacteria', 'Haemophilus_sp1': 'Proteobacteria',
        'Neisseria_sp1': 'Proteobacteria', 'Pseudomonas_sp1': 'Proteobacteria',
        'Acinetobacter_sp1': 'Proteobacteria',
        'Fusobacterium_sp1': 'Fusobacteria', 'Fusobacterium_sp2': 'Fusobacteria',
        'Porphyromonas_sp1': 'Bacteroidetes', 'Tannerella_sp1': 'Bacteroidetes',
        'Treponema_sp1': 'Spirochaetes',
    }
    # Add more taxa to reach n_taxa
    existing = len(taxa_info)
    phyla = ['Firmicutes', 'Bacteroidetes', 'Actinobacteria', 'Proteobacteria',
             'Verrucomicrobia', 'Tenericutes']
    for i in range(n_taxa - existing):
        phylum = phyla[i % len(phyla)]
        taxa_info[f'Taxon_{i+1:03d}'] = phylum
    
    taxa_names = list(taxa_info.keys())[:n_taxa]
    
    # Define characteristic community profiles for each body site
    # (relative abundance templates -- Dirichlet concentration parameters)
    profiles = {
        'Gut': np.array([15, 10, 8, 5, 4, 12, 8, 6, 5, 3, 2, 1, 1, 1, 0.5,
                         0.2, 0.1, 1, 1, 0.5, 6, 4, 0.5, 0.1, 0.1, 0.1, 2,
                         0.5, 0.1, 0.1, 0.1, 0.5, 0.2, 0.5, 0.2, 0.1] +
                        [0.3] * (n_taxa - 36)),
        'Skin': np.array([0.5, 0.3, 0.2, 0.1, 0.1, 0.5, 0.3, 0.2, 0.5, 0.3,
                          0.5, 0.3, 1, 0.5, 0.2, 15, 12, 0.3, 0.1, 0.1,
                          0.5, 0.3, 10, 8, 8, 6, 0.5, 0.3, 0.2, 1, 2,
                          0.2, 0.1, 0.5, 0.2, 0.1] +
                         [0.3] * (n_taxa - 36)),
        'Oral': np.array([1, 0.5, 3, 2, 0.3, 1, 0.5, 0.3, 0.5, 0.2,
                          2, 1, 12, 10, 8, 0.5, 0.3, 0.5, 5, 3,
                          0.5, 0.3, 0.5, 0.3, 0.2, 0.1, 1, 6, 5, 1,
                          0.5, 8, 5, 3, 2, 1] +
                         [0.3] * (n_taxa - 36)),
    }
    
    # Generate samples
    sample_names = []
    sample_sites = []
    sample_subjects = []
    counts_data = []
    
    for site in sites:
        for i in range(n_samples_per_site):
            sample_name = f"{site}_{i+1}"
            sample_names.append(sample_name)
            sample_sites.append(site)
            sample_subjects.append(f"Subject_{(i % 4) + 1}")
            
            total_reads = np.random.randint(*total_reads_range)
            # Dirichlet-multinomial: sample relative abundances, then counts
            rel_abund = np.random.dirichlet(profiles[site][:n_taxa] + 0.1)
            sample_counts = np.random.multinomial(total_reads, rel_abund)
            counts_data.append(sample_counts)
    
    feature_table = pd.DataFrame(
        np.array(counts_data).T,
        index=taxa_names,
        columns=sample_names
    )
    
    sample_metadata = pd.DataFrame({
        'body_site': sample_sites,
        'subject': sample_subjects,
    }, index=sample_names)
    
    taxonomy = pd.DataFrame({
        'Phylum': [taxa_info.get(t, 'Unknown') for t in taxa_names]
    }, index=taxa_names)
    
    return feature_table, sample_metadata, taxonomy


# Generate the dataset
feature_table, sample_metadata, taxonomy = simulate_microbiome_data()

print(f"Feature table: {feature_table.shape[0]} taxa x {feature_table.shape[1]} samples")
print(f"\nSample metadata:")
print(sample_metadata.head(6))
print(f"\nSamples per body site:")
print(sample_metadata['body_site'].value_counts())
print(f"\nLibrary sizes range: {feature_table.sum(axis=0).min()} - {feature_table.sum(axis=0).max()}")

In [ ]:
# Preview the feature table
print("Feature table (first 8 taxa, first 6 samples):")
feature_table.iloc[:8, :6]

## 6. Alpha Diversity: Who Is There? How Many?

**Alpha diversity** measures the diversity *within* a single sample. Two key aspects:

- **Richness**: How many different types are present?
- **Evenness**: How evenly are abundances distributed?

A sample with 100 species all at equal abundance is more diverse (in terms of evenness) than one with 100 species where a single species dominates 99% of reads.

### Qualitative vs. Quantitative

- **Qualitative** metrics consider only presence/absence (e.g., observed species)
- **Quantitative** metrics also consider relative abundance (e.g., Shannon, Simpson)

### Phylogenetic vs. Non-phylogenetic

- **Non-phylogenetic**: treat all taxa as equally related
- **Phylogenetic**: incorporate the evolutionary tree (e.g., Faith's PD)

### 6.1 Observed Species (Richness)

The simplest metric: count the number of features (OTUs/ASVs) with count > 0.

In [ ]:
def observed_species(counts):
    """Count the number of non-zero features in a sample."""
    return np.sum(counts > 0)

# Compute for each sample
obs_sp = feature_table.apply(observed_species, axis=0)

print("Observed species per sample:")
for site in ['Gut', 'Skin', 'Oral']:
    site_samples = sample_metadata[sample_metadata['body_site'] == site].index
    site_values = obs_sp[site_samples]
    print(f"  {site}: mean={site_values.mean():.1f}, range={site_values.min()}-{site_values.max()}")

### 6.2 Shannon Diversity Index

The Shannon index ($H'$) accounts for both richness and evenness:

$$H' = -\sum_{i=1}^{S} p_i \ln(p_i)$$

where $p_i$ is the relative abundance of taxon $i$, and $S$ is the number of taxa.

- Higher values = more diverse
- $H' = 0$ when only one species is present
- Maximum when all species are equally abundant: $H'_{max} = \ln(S)$

In [ ]:
def shannon_diversity(counts):
    """Compute Shannon diversity index H'."""
    counts = np.array(counts, dtype=float)
    counts = counts[counts > 0]  # exclude zeros
    proportions = counts / counts.sum()
    return -np.sum(proportions * np.log(proportions))

# Test with simple examples
print("Shannon index examples:")
print(f"  [100, 0, 0, 0]:  H' = {shannon_diversity([100, 0, 0, 0]):.3f}  (one species dominates)")
print(f"  [25, 25, 25, 25]: H' = {shannon_diversity([25, 25, 25, 25]):.3f}  (perfectly even)")
print(f"  [90, 5, 3, 2]:   H' = {shannon_diversity([90, 5, 3, 2]):.3f}  (uneven)")
print(f"  Max for S=4:     H' = {np.log(4):.3f}")

### 6.3 Simpson Diversity Index

Simpson's index measures the probability that two randomly chosen reads belong to the same taxon:

$$D = \sum_{i=1}^{S} p_i^2$$

This ranges from 0 to 1, where 1 means all reads are the same species. By convention, we often report the complement:

$$1 - D = 1 - \sum_{i=1}^{S} p_i^2$$

where higher values mean more diversity.

In [ ]:
def simpson_diversity(counts):
    """Compute Simpson's diversity (1 - D)."""
    counts = np.array(counts, dtype=float)
    counts = counts[counts > 0]
    proportions = counts / counts.sum()
    return 1 - np.sum(proportions ** 2)

print("Simpson (1-D) examples:")
print(f"  [100, 0, 0, 0]:  1-D = {simpson_diversity([100, 0, 0, 0]):.3f}  (no diversity)")
print(f"  [25, 25, 25, 25]: 1-D = {simpson_diversity([25, 25, 25, 25]):.3f}  (max diversity)")
print(f"  [90, 5, 3, 2]:   1-D = {simpson_diversity([90, 5, 3, 2]):.3f}  (low diversity)")

### 6.4 Faith's Phylogenetic Diversity (PD)

Faith's PD is the sum of the branch lengths in the phylogenetic tree that are represented by the taxa observed in a sample.

$$PD = \sum_{\text{branches observed}} \text{branch\_length}$$

A sample containing distantly related organisms has higher PD than one with only closely related organisms, even if both have the same number of species. This is the key advantage of phylogenetic metrics: they incorporate evolutionary relationships.

We will simulate a simple phylogenetic tree to demonstrate.

In [ ]:
def simulate_tree_distances(taxa_names, taxonomy):
    """
    Simulate phylogenetic branch lengths based on taxonomy.
    Taxa in the same phylum are closer; different phyla are more distant.
    Returns a dictionary mapping each taxon to its branch length contribution.
    """
    # Assign branch lengths based on phylum grouping
    phyla = taxonomy['Phylum'].unique()
    phylum_branch = {p: np.random.uniform(0.3, 0.6) for p in phyla}
    
    branch_lengths = {}
    for taxon in taxa_names:
        phylum = taxonomy.loc[taxon, 'Phylum']
        # Tip branch length + shared phylum branch
        branch_lengths[taxon] = {
            'tip': np.random.uniform(0.05, 0.2),
            'phylum_branch': phylum_branch[phylum]
        }
    
    return branch_lengths, phylum_branch


def faiths_pd(counts, branch_lengths, phylum_branch, taxonomy):
    """
    Simplified Faith's PD calculation.
    Sums tip branches for observed taxa + shared phylum branches for observed phyla.
    """
    observed_taxa = [t for t, c in zip(counts.index, counts.values) if c > 0]
    
    # Sum tip branches
    pd_value = sum(branch_lengths[t]['tip'] for t in observed_taxa)
    
    # Add phylum-level branches for observed phyla
    observed_phyla = set(taxonomy.loc[t, 'Phylum'] for t in observed_taxa)
    pd_value += sum(phylum_branch[p] for p in observed_phyla)
    
    return pd_value


branch_lengths, phylum_branch = simulate_tree_distances(feature_table.index, taxonomy)

pd_values = {}
for sample in feature_table.columns:
    pd_values[sample] = faiths_pd(feature_table[sample], branch_lengths,
                                   phylum_branch, taxonomy)

pd_series = pd.Series(pd_values)

print("Faith's PD per body site:")
for site in ['Gut', 'Skin', 'Oral']:
    site_samples = sample_metadata[sample_metadata['body_site'] == site].index
    vals = pd_series[site_samples]
    print(f"  {site}: mean={vals.mean():.2f}, std={vals.std():.2f}")

### 6.5 Comparing Alpha Diversity Across Groups

In [ ]:
# Compute all alpha diversity metrics for all samples
alpha_df = pd.DataFrame({
    'Observed Species': feature_table.apply(observed_species, axis=0),
    'Shannon': feature_table.apply(shannon_diversity, axis=0),
    'Simpson (1-D)': feature_table.apply(simpson_diversity, axis=0),
    "Faith's PD": pd_series,
    'body_site': sample_metadata['body_site']
})

# Plot
metrics = ['Observed Species', 'Shannon', 'Simpson (1-D)', "Faith's PD"]
site_colors = {'Gut': '#2196F3', 'Skin': '#FF9800', 'Oral': '#4CAF50'}
sites = ['Gut', 'Skin', 'Oral']

fig, axes = plt.subplots(1, 4, figsize=(16, 5))

for ax, metric in zip(axes, metrics):
    data_per_site = [alpha_df[alpha_df['body_site'] == s][metric].values for s in sites]
    
    bp = ax.boxplot(data_per_site, patch_artist=True, widths=0.6)
    for patch, site in zip(bp['boxes'], sites):
        patch.set_facecolor(site_colors[site])
        patch.set_alpha(0.7)
    
    ax.set_xticklabels(sites)
    ax.set_ylabel(metric)
    ax.set_title(metric)
    ax.grid(True, alpha=0.3)

plt.suptitle('Alpha Diversity by Body Site', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# Statistical test (Kruskal-Wallis)
print("\nKruskal-Wallis tests (body site differences):")
for metric in metrics:
    groups = [alpha_df[alpha_df['body_site'] == s][metric].values for s in sites]
    stat, pval = stats.kruskal(*groups)
    print(f"  {metric}: H={stat:.2f}, p={pval:.4f} {'*' if pval < 0.05 else ''}")

## 7. Rarefaction Curves

A fundamental challenge in microbial diversity analysis: **have we sequenced deeply enough to capture the true diversity?**

Rarefaction curves address this by subsampling reads at increasing depths and computing diversity at each depth. If the curve plateaus, we have captured most of the diversity. If it is still rising steeply, we would discover more taxa with deeper sequencing.

### Even Sampling (Rarefying)

Samples with different sequencing depths are not directly comparable for alpha diversity (more reads = more taxa observed). **Rarefaction** subsamples all samples to the same depth to enable fair comparison. The rarefaction depth is typically set to the minimum library size (or slightly below to avoid discarding too many samples).

In [ ]:
def rarefaction_curve(counts, depths, n_iterations=10):
    """
    Compute rarefaction curve for a single sample.
    At each depth, subsample reads and compute observed species.
    """
    counts = np.array(counts, dtype=int)
    total = counts.sum()
    
    # Expand counts to individual reads for subsampling
    reads = np.repeat(np.arange(len(counts)), counts)
    
    results = []
    for depth in depths:
        if depth > total:
            results.append(np.nan)
            continue
        
        obs_values = []
        for _ in range(n_iterations):
            subsample = np.random.choice(reads, size=depth, replace=False)
            n_observed = len(np.unique(subsample))
            obs_values.append(n_observed)
        results.append(np.mean(obs_values))
    
    return results


# Compute rarefaction curves
min_depth = int(feature_table.sum(axis=0).min())
depths = np.linspace(100, min_depth, 20, dtype=int)

fig, ax = plt.subplots(figsize=(10, 6))

for sample in feature_table.columns:
    site = sample_metadata.loc[sample, 'body_site']
    color = site_colors[site]
    
    curve = rarefaction_curve(feature_table[sample].values, depths)
    ax.plot(depths, curve, color=color, alpha=0.5, linewidth=1)

# Add legend
for site, color in site_colors.items():
    ax.plot([], [], color=color, linewidth=2, label=site)

ax.set_xlabel('Sequencing Depth (number of reads)')
ax.set_ylabel('Observed Species')
ax.set_title('Rarefaction Curves')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Minimum library size: {min_depth} reads")
print("If curves are plateauing, sequencing depth is sufficient.")

## 8. Beta Diversity: How Similar Are Communities?

**Beta diversity** measures the difference *between* pairs of samples. The result is a **distance matrix** where each entry is the dissimilarity between two samples.

Just as with alpha diversity, beta diversity metrics can be:
- **Qualitative** (presence/absence) vs. **Quantitative** (abundance)
- **Non-phylogenetic** vs. **Phylogenetic**

| Metric | Phylogenetic? | Quantitative? | Range |
|--------|:---:|:---:|:---:|
| Jaccard | No | No | [0, 1] |
| Bray-Curtis | No | Yes | [0, 1] |
| Unweighted UniFrac | Yes | No | [0, 1] |
| Weighted UniFrac | Yes | Yes | [0, 1] |

### 8.1 Bray-Curtis Dissimilarity

Bray-Curtis is the most widely used non-phylogenetic beta diversity metric:

$$BC_{jk} = \frac{\sum_{i} |X_{ij} - X_{ik}|}{\sum_{i} (X_{ij} + X_{ik})}$$

where $X_{ij}$ is the count of taxon $i$ in sample $j$.

- 0 = identical communities
- 1 = completely different communities (no shared taxa)

In [ ]:
def bray_curtis_distance(sample1, sample2):
    """
    Compute Bray-Curtis dissimilarity between two samples.
    """
    s1 = np.array(sample1, dtype=float)
    s2 = np.array(sample2, dtype=float)
    numerator = np.sum(np.abs(s1 - s2))
    denominator = np.sum(s1 + s2)
    if denominator == 0:
        return 0.0
    return numerator / denominator


def compute_distance_matrix(table, distance_fn):
    """Compute a pairwise distance matrix for all samples."""
    samples = table.columns
    n = len(samples)
    dm = np.zeros((n, n))
    
    for i in range(n):
        for j in range(i + 1, n):
            d = distance_fn(table.iloc[:, i].values, table.iloc[:, j].values)
            dm[i, j] = d
            dm[j, i] = d
    
    return pd.DataFrame(dm, index=samples, columns=samples)


# Compute Bray-Curtis distance matrix
bc_dm = compute_distance_matrix(feature_table, bray_curtis_distance)

print("Bray-Curtis distance matrix (first 6 samples):")
print(bc_dm.iloc[:6, :6].round(3))

### 8.2 UniFrac Distances

UniFrac distances incorporate the phylogenetic tree. The key idea: two communities that differ in closely related taxa are more similar than two communities that differ in distantly related taxa.

**Unweighted UniFrac** (qualitative):

$$U_{AB} = \frac{\text{unique branch length}}{\text{total observed branch length}}$$

where "unique" means branches leading only to taxa in sample A or only to taxa in sample B.

**Weighted UniFrac** (quantitative): Also considers the abundance of taxa at each branch, not just presence/absence.

Since full UniFrac requires a phylogenetic tree, here we implement a simplified version using our simulated phylogenetic distances.

In [ ]:
def simplified_unifrac(sample1, sample2, taxonomy):
    """
    Simplified unweighted UniFrac approximation.
    Uses phylum-level grouping as a proxy for phylogenetic distance.
    Taxa in different phyla contribute more to distance.
    """
    s1 = np.array(sample1, dtype=float)
    s2 = np.array(sample2, dtype=float)
    taxa = sample1.index
    
    present_in_1 = set(taxa[s1 > 0])
    present_in_2 = set(taxa[s2 > 0])
    
    # Unique taxa (in one but not the other)
    unique_to_1 = present_in_1 - present_in_2
    unique_to_2 = present_in_2 - present_in_1
    shared = present_in_1 & present_in_2
    all_observed = present_in_1 | present_in_2
    
    if len(all_observed) == 0:
        return 0.0
    
    # Weight unique taxa by phylogenetic depth
    unique_phyla_1 = set(taxonomy.loc[t, 'Phylum'] for t in unique_to_1)
    unique_phyla_2 = set(taxonomy.loc[t, 'Phylum'] for t in unique_to_2)
    shared_phyla = set(taxonomy.loc[t, 'Phylum'] for t in shared)
    
    # Tip-level uniqueness
    tip_unique = len(unique_to_1) + len(unique_to_2)
    tip_total = len(all_observed)
    
    # Phylum-level uniqueness (phyla only in one sample)
    phylum_unique = len((unique_phyla_1 - shared_phyla) | (unique_phyla_2 - shared_phyla))
    phylum_total = len(unique_phyla_1 | unique_phyla_2 | shared_phyla)
    
    # Combine tip and phylum contributions
    if tip_total == 0:
        return 0.0
    return 0.7 * (tip_unique / tip_total) + 0.3 * (phylum_unique / max(1, phylum_total))


# Compute simplified UniFrac distance matrix
n_samples = len(feature_table.columns)
uf_matrix = np.zeros((n_samples, n_samples))

for i in range(n_samples):
    for j in range(i + 1, n_samples):
        d = simplified_unifrac(feature_table.iloc[:, i], feature_table.iloc[:, j], taxonomy)
        uf_matrix[i, j] = d
        uf_matrix[j, i] = d

uf_dm = pd.DataFrame(uf_matrix, index=feature_table.columns, columns=feature_table.columns)

print("Simplified UniFrac distance matrix (first 6 samples):")
print(uf_dm.iloc[:6, :6].round(3))

### 8.3 Visualizing the Distance Matrix

In [ ]:
# Heatmap of Bray-Curtis distances
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Sort samples by body site for cleaner visualization
sorted_samples = sample_metadata.sort_values('body_site').index

for ax, dm, title in [(axes[0], bc_dm, 'Bray-Curtis'),
                       (axes[1], uf_dm, 'Simplified UniFrac')]:
    dm_sorted = dm.loc[sorted_samples, sorted_samples]
    im = ax.imshow(dm_sorted, cmap='viridis', aspect='auto')
    ax.set_title(f'{title} Distance Matrix')
    
    # Add body site color bar on top
    site_labels = sample_metadata.loc[sorted_samples, 'body_site']
    boundaries = []
    prev_site = None
    for idx, site in enumerate(site_labels):
        if site != prev_site:
            boundaries.append((idx, site))
            prev_site = site
    
    for start_idx, site in boundaries:
        ax.axhline(start_idx - 0.5, color='white', linewidth=1)
        ax.axvline(start_idx - 0.5, color='white', linewidth=1)
    
    ax.set_xticks([])
    ax.set_yticks([b[0] + 3 for b in boundaries])
    ax.set_yticklabels([b[1] for b in boundaries])
    plt.colorbar(im, ax=ax, shrink=0.7, label='Distance')

plt.tight_layout()
plt.show()

## 9. Ordination: Visualizing Community Differences

Distance matrices are hard to interpret directly. **Ordination** methods reduce the dimensionality of the distance matrix to 2 or 3 dimensions for visualization.

### Principal Coordinates Analysis (PCoA)

PCoA (also called Classical MDS) finds coordinates in a low-dimensional space that best preserve the original pairwise distances. It is the standard ordination method for microbiome data.

Key properties:
- Axes are ordered by the amount of variation they explain
- PC1 captures the most variation, PC2 the second most, etc.
- The proportion of variance explained by each axis is informative

### NMDS (Non-metric Multidimensional Scaling)

NMDS preserves the **rank order** of distances rather than the exact values. This makes it more robust to non-linear relationships in the data.

- Reports a **stress** value: lower stress = better fit (stress < 0.1 is good, < 0.2 is acceptable)
- Axes are arbitrary (no "% variance explained")
- Better when the relationship between distances is non-linear

In [ ]:
def pcoa(distance_matrix):
    """
    Perform Principal Coordinates Analysis (classical MDS).
    Returns coordinates and proportion of variance explained.
    """
    dm = distance_matrix.values
    n = dm.shape[0]
    
    # Double centering
    H = np.eye(n) - np.ones((n, n)) / n
    B = -0.5 * H @ (dm ** 2) @ H
    
    # Eigendecomposition
    eigenvalues, eigenvectors = np.linalg.eigh(B)
    
    # Sort by decreasing eigenvalue
    idx = np.argsort(eigenvalues)[::-1]
    eigenvalues = eigenvalues[idx]
    eigenvectors = eigenvectors[:, idx]
    
    # Keep only positive eigenvalues
    positive = eigenvalues > 0
    eigenvalues = eigenvalues[positive]
    eigenvectors = eigenvectors[:, positive]
    
    # Coordinates
    coordinates = eigenvectors * np.sqrt(eigenvalues)
    
    # Proportion of variance explained
    prop_explained = eigenvalues / eigenvalues.sum()
    
    coord_df = pd.DataFrame(
        coordinates[:, :3],
        index=distance_matrix.index,
        columns=['PC1', 'PC2', 'PC3']
    )
    
    return coord_df, prop_explained


# PCoA on Bray-Curtis
bc_coords, bc_explained = pcoa(bc_dm)

print("PCoA (Bray-Curtis):")
print(f"  PC1 explains {bc_explained[0]*100:.1f}% of variation")
print(f"  PC2 explains {bc_explained[1]*100:.1f}% of variation")
print(f"  PC3 explains {bc_explained[2]*100:.1f}% of variation")

In [ ]:
# PCoA plot (Bray-Curtis)
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Plot 1: PCoA on Bray-Curtis
for site in sites:
    mask = sample_metadata['body_site'] == site
    site_samples = sample_metadata[mask].index
    axes[0].scatter(
        bc_coords.loc[site_samples, 'PC1'],
        bc_coords.loc[site_samples, 'PC2'],
        c=site_colors[site], s=80, edgecolors='black',
        label=site, alpha=0.8, zorder=3
    )

axes[0].set_xlabel(f'PC1 ({bc_explained[0]*100:.1f}%)')
axes[0].set_ylabel(f'PC2 ({bc_explained[1]*100:.1f}%)')
axes[0].set_title('PCoA: Bray-Curtis')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot 2: PCoA on UniFrac
uf_coords, uf_explained = pcoa(uf_dm)

for site in sites:
    mask = sample_metadata['body_site'] == site
    site_samples = sample_metadata[mask].index
    axes[1].scatter(
        uf_coords.loc[site_samples, 'PC1'],
        uf_coords.loc[site_samples, 'PC2'],
        c=site_colors[site], s=80, edgecolors='black',
        label=site, alpha=0.8, zorder=3
    )

axes[1].set_xlabel(f'PC1 ({uf_explained[0]*100:.1f}%)')
axes[1].set_ylabel(f'PC2 ({uf_explained[1]*100:.1f}%)')
axes[1].set_title('PCoA: Simplified UniFrac')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Beta Diversity Ordination', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# NMDS on Bray-Curtis
nmds = MDS(n_components=2, dissimilarity='precomputed', metric=False,
           random_state=42, n_init=10, max_iter=300, normalized_stress='auto')
nmds_coords = nmds.fit_transform(bc_dm.values)

fig, ax = plt.subplots(figsize=(8, 6))

for site in sites:
    mask = sample_metadata['body_site'] == site
    idx = np.where(mask.values)[0]
    ax.scatter(
        nmds_coords[idx, 0], nmds_coords[idx, 1],
        c=site_colors[site], s=80, edgecolors='black',
        label=site, alpha=0.8, zorder=3
    )

ax.set_xlabel('NMDS1')
ax.set_ylabel('NMDS2')
ax.set_title(f'NMDS: Bray-Curtis (stress={nmds.stress_:.3f})')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 10. Statistical Testing on Distance Matrices

Ordination plots give visual impressions, but we need statistical tests to determine if group differences are significant.

### PERMANOVA (Permutational Multivariate Analysis of Variance)

PERMANOVA (Anderson, 2001) tests whether group centroids in multivariate space differ. It works directly on the distance matrix:

1. Partition total variation into within-group and between-group components
2. Compute an F-statistic (ratio of between/within group variation)
3. Permute group labels many times to build a null distribution
4. The p-value is the fraction of permuted F-statistics >= observed

### ANOSIM (Analysis of Similarities)

ANOSIM tests whether between-group distances are greater than within-group distances.

$$R = \frac{\bar{r}_B - \bar{r}_W}{N(N-1)/4}$$

where $\bar{r}_B$ and $\bar{r}_W$ are mean ranks of between- and within-group distances.

- R = 1: all between-group distances are greater than within-group
- R = 0: no difference between groups
- R < 0: within-group distances are larger (unusual)

In [ ]:
def permanova(dm, grouping, n_permutations=999):
    """
    Simplified PERMANOVA implementation.
    Tests whether group centroids differ based on a distance matrix.
    """
    dm_values = dm.values
    samples = dm.index
    groups = grouping.loc[samples].values
    unique_groups = np.unique(groups)
    n = len(samples)
    
    def compute_f_statistic(group_labels):
        # Total sum of squared distances
        ss_total = np.sum(dm_values ** 2) / n
        
        # Within-group sum of squares
        ss_within = 0
        for g in unique_groups:
            mask = group_labels == g
            n_g = mask.sum()
            if n_g > 1:
                within_dm = dm_values[np.ix_(mask, mask)]
                ss_within += np.sum(within_dm ** 2) / (2 * n_g)
        
        ss_between = ss_total - ss_within
        
        df_between = len(unique_groups) - 1
        df_within = n - len(unique_groups)
        
        if df_within == 0 or ss_within == 0:
            return 0.0
        
        f_stat = (ss_between / df_between) / (ss_within / df_within)
        return f_stat
    
    # Observed F-statistic
    observed_f = compute_f_statistic(groups)
    
    # Permutation test
    count_geq = 0
    for _ in range(n_permutations):
        permuted_groups = np.random.permutation(groups)
        perm_f = compute_f_statistic(permuted_groups)
        if perm_f >= observed_f:
            count_geq += 1
    
    p_value = (count_geq + 1) / (n_permutations + 1)
    
    return observed_f, p_value


def anosim(dm, grouping, n_permutations=999):
    """
    Simplified ANOSIM implementation.
    Tests if between-group distances > within-group distances.
    """
    dm_values = dm.values
    samples = dm.index
    groups = grouping.loc[samples].values
    n = len(samples)
    
    # Get all pairwise distances as ranks
    distances = []
    labels = []  # 'within' or 'between'
    for i in range(n):
        for j in range(i + 1, n):
            distances.append(dm_values[i, j])
            labels.append('within' if groups[i] == groups[j] else 'between')
    
    distances = np.array(distances)
    labels = np.array(labels)
    ranks = stats.rankdata(distances)
    
    def compute_r(rank_values, lab):
        r_within = rank_values[lab == 'within'].mean()
        r_between = rank_values[lab == 'between'].mean()
        n_pairs = len(rank_values)
        return (r_between - r_within) / (n_pairs / 2)
    
    observed_r = compute_r(ranks, labels)
    
    count_geq = 0
    for _ in range(n_permutations):
        perm_groups = np.random.permutation(groups)
        perm_labels = []
        for i in range(n):
            for j in range(i + 1, n):
                perm_labels.append('within' if perm_groups[i] == perm_groups[j] else 'between')
        perm_labels = np.array(perm_labels)
        perm_r = compute_r(ranks, perm_labels)
        if perm_r >= observed_r:
            count_geq += 1
    
    p_value = (count_geq + 1) / (n_permutations + 1)
    return observed_r, p_value


# Run PERMANOVA
print("PERMANOVA (Bray-Curtis, body site):")
f_stat, p_val = permanova(bc_dm, sample_metadata['body_site'])
print(f"  F-statistic: {f_stat:.3f}")
print(f"  p-value: {p_val:.4f}")
print(f"  Significant: {'Yes' if p_val < 0.05 else 'No'}")

print("\nANOSIM (Bray-Curtis, body site):")
r_stat, p_val = anosim(bc_dm, sample_metadata['body_site'])
print(f"  R-statistic: {r_stat:.3f}")
print(f"  p-value: {p_val:.4f}")
print(f"  Significant: {'Yes' if p_val < 0.05 else 'No'}")

### Within vs. Between Group Distances

In [ ]:
# Visualize within vs. between group distances
def within_between_distances(dm, grouping):
    """Extract within-group and between-group distances."""
    samples = dm.index
    groups = grouping.loc[samples]
    within = []
    between = []
    
    for i in range(len(samples)):
        for j in range(i + 1, len(samples)):
            d = dm.iloc[i, j]
            if groups.iloc[i] == groups.iloc[j]:
                within.append(d)
            else:
                between.append(d)
    
    return within, between


within_dists, between_dists = within_between_distances(bc_dm, sample_metadata['body_site'])

fig, ax = plt.subplots(figsize=(8, 5))

bp = ax.boxplot([within_dists, between_dists], patch_artist=True, widths=0.5)
bp['boxes'][0].set_facecolor('#66BB6A')
bp['boxes'][1].set_facecolor('#EF5350')

ax.set_xticklabels(['Within body site', 'Between body sites'])
ax.set_ylabel('Bray-Curtis Distance')
ax.set_title('Within vs. Between Group Distances')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Mean within-group distance:  {np.mean(within_dists):.3f}")
print(f"Mean between-group distance: {np.mean(between_dists):.3f}")

## 11. Visualization: Taxonomic Bar Plots

Stacked bar plots showing relative abundance at the phylum (or genus) level are the most common way to visualize community composition.

In [ ]:
# Aggregate to phylum level
phylum_table = feature_table.copy()
phylum_table['Phylum'] = taxonomy['Phylum']
phylum_table = phylum_table.groupby('Phylum').sum()

# Convert to relative abundance
rel_abund = phylum_table.div(phylum_table.sum(axis=0), axis=1)

# Sort samples by body site
sorted_samples = sample_metadata.sort_values('body_site').index
rel_abund = rel_abund[sorted_samples]

# Sort phyla by mean abundance
phylum_order = rel_abund.mean(axis=1).sort_values(ascending=False).index
rel_abund = rel_abund.loc[phylum_order]

# Color palette for phyla
phylum_colors = {
    'Firmicutes': '#1f77b4',
    'Bacteroidetes': '#ff7f0e',
    'Actinobacteria': '#2ca02c',
    'Proteobacteria': '#d62728',
    'Fusobacteria': '#9467bd',
    'Spirochaetes': '#8c564b',
    'Verrucomicrobia': '#e377c2',
    'Tenericutes': '#7f7f7f',
}

fig, ax = plt.subplots(figsize=(16, 6))

bottom = np.zeros(len(sorted_samples))
for phylum in rel_abund.index:
    color = phylum_colors.get(phylum, '#bcbd22')
    ax.bar(range(len(sorted_samples)), rel_abund.loc[phylum].values,
           bottom=bottom, color=color, label=phylum, width=0.9)
    bottom += rel_abund.loc[phylum].values

# Add body site labels
site_boundaries = []
prev_site = None
for i, sample in enumerate(sorted_samples):
    site = sample_metadata.loc[sample, 'body_site']
    if site != prev_site:
        site_boundaries.append(i)
        prev_site = site

for b in site_boundaries[1:]:
    ax.axvline(b - 0.5, color='black', linewidth=2)

ax.set_xlabel('Sample')
ax.set_ylabel('Relative Abundance')
ax.set_title('Phylum-Level Community Composition')
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9)
ax.set_xticks(range(len(sorted_samples)))
ax.set_xticklabels(sorted_samples, rotation=90, fontsize=7)
ax.set_ylim(0, 1)

plt.tight_layout()
plt.show()

### Mean Composition by Body Site

In [ ]:
# Mean relative abundance per body site
fig, axes = plt.subplots(1, 3, figsize=(14, 5))

for ax, site in zip(axes, sites):
    site_samples = sample_metadata[sample_metadata['body_site'] == site].index
    mean_abund = rel_abund[site_samples].mean(axis=1).sort_values(ascending=True)
    
    # Only show phyla with > 1% mean abundance
    mean_abund = mean_abund[mean_abund > 0.01]
    
    colors = [phylum_colors.get(p, '#bcbd22') for p in mean_abund.index]
    ax.barh(range(len(mean_abund)), mean_abund.values, color=colors, edgecolor='black')
    ax.set_yticks(range(len(mean_abund)))
    ax.set_yticklabels(mean_abund.index, fontsize=9)
    ax.set_xlabel('Mean Relative Abundance')
    ax.set_title(f'{site} (n={len(site_samples)})')
    ax.set_xlim(0, max(0.5, mean_abund.max() * 1.1))

plt.suptitle('Mean Phylum Composition by Body Site', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 12. Hierarchical Clustering

Another way to visualize sample relationships from a distance matrix is hierarchical clustering, which produces a dendrogram (tree).

In [ ]:
# Hierarchical clustering of Bray-Curtis distances
from scipy.spatial.distance import squareform

condensed = squareform(bc_dm.values)
Z = linkage(condensed, method='average')  # UPGMA

fig, ax = plt.subplots(figsize=(14, 6))

# Color labels by body site
label_colors = {s: site_colors[sample_metadata.loc[s, 'body_site']]
                for s in bc_dm.index}

dendro = dendrogram(Z, labels=bc_dm.index.tolist(), leaf_rotation=90,
                    ax=ax, leaf_font_size=8)

# Color the x-axis labels
xlabels = ax.get_xticklabels()
for label in xlabels:
    label.set_color(label_colors.get(label.get_text(), 'black'))

ax.set_ylabel('Bray-Curtis Distance')
ax.set_title('UPGMA Clustering of Samples (Bray-Curtis)')

# Legend
for site, color in site_colors.items():
    ax.plot([], [], 's', color=color, label=site, markersize=10)
ax.legend(loc='upper right')

plt.tight_layout()
plt.show()

## 13. Tools Ecosystem

### QIIME2

[QIIME2](https://qiime2.org/) is the most widely used amplicon sequencing analysis platform:
- Complete pipeline from raw reads to publication-quality figures
- Plugin architecture (DADA2, VSEARCH, etc.)
- Provenance tracking (every analysis step is recorded)
- Interactive visualization (qiime2 view)

```bash
# Example QIIME2 workflow
qiime dada2 denoise-paired \
    --i-demultiplexed-seqs reads.qza \
    --p-trunc-len-f 230 --p-trunc-len-r 200 \
    --o-table table.qza \
    --o-representative-sequences rep-seqs.qza

qiime diversity core-metrics-phylogenetic \
    --i-table table.qza \
    --i-phylogeny tree.qza \
    --p-sampling-depth 10000 \
    --m-metadata-file metadata.tsv \
    --output-dir diversity-results/
```

### mothur

[mothur](https://mothur.org/) is another major platform, with a command-line interface and extensive documentation. Historically focused on OTU-based approaches.

### phyloseq (R)

[phyloseq](https://joey711.github.io/phyloseq/) is an R/Bioconductor package for microbiome analysis and visualization. It integrates with ggplot2 for beautiful plots.

```r
library(phyloseq)
library(ggplot2)

# Load data
ps <- phyloseq(otu_table, sample_data, tax_table, phy_tree)

# Alpha diversity
plot_richness(ps, x="body_site", measures=c("Shannon", "Simpson"))

# Beta diversity (PCoA)
ordinate(ps, method="PCoA", distance="bray") %>%
  plot_ordination(ps, ., color="body_site")
```

### Python Tools

| Package | Use |
|---------|-----|
| scikit-bio | Distance matrices, ordination, diversity metrics |
| biom-format | BIOM table I/O |
| emperor | Interactive 3D ordination plots |
| calour | Microbiome analysis and visualization |
| qiime2 Python API | Programmatic access to QIIME2 |

## 14. Summary of Key Concepts

| Concept | Key Point |
|---------|-----------|
| 16S rRNA | Universal bacterial marker with conserved + variable regions |
| OTU vs ASV | ASVs (DADA2) are now preferred over 97% OTUs |
| Alpha diversity | Within-sample: richness (observed species), evenness (Shannon, Simpson) |
| Rarefaction | Subsample to equal depth before comparing alpha diversity |
| Beta diversity | Between-sample: Bray-Curtis (abundance), UniFrac (phylogenetic) |
| Ordination | PCoA or NMDS to visualize distance matrices in 2D |
| Statistical testing | PERMANOVA and ANOSIM for group differences |
| Stacked bar plots | Show community composition at phylum/genus level |

## Exercises

### Exercise 1: Shannon Evenness

Implement **Pielou's evenness** index:

$$J' = \frac{H'}{H'_{\max}} = \frac{H'}{\ln(S)}$$

where $S$ is the number of observed species and $H'$ is the Shannon index.

Compute evenness for all samples and compare across body sites. Which body site has the most even communities?

```python
def pielou_evenness(counts):
    """Compute Pielou's evenness index J'."""
    # Your code here
    pass
```

In [ ]:
# Exercise 1: your code here


### Exercise 2: Jaccard Distance

Implement the **Jaccard distance** (qualitative, non-phylogenetic):

$$J_{dist}(A, B) = 1 - \frac{|A \cap B|}{|A \cup B|}$$

where $A$ and $B$ are the sets of taxa present in each sample.

1. Compute the Jaccard distance matrix
2. Run PCoA on it
3. Compare the PCoA plot with the Bray-Curtis PCoA. Do they tell the same story?

In [ ]:
# Exercise 2: your code here


### Exercise 3: Rarefaction and Alpha Diversity

1. Rarefy (subsample) all samples to the minimum library size
2. Recompute Shannon and observed species on the rarefied data
3. Compare the rarefied results with the un-rarefied results
4. Does rarefaction change your conclusions about which body site is most diverse?

```python
def rarefy(counts, depth):
    """Subsample counts to a specified depth."""
    # Your code here
    pass
```

In [ ]:
# Exercise 3: your code here


### Exercise 4: Pairwise PERMANOVA

The PERMANOVA above tests whether *any* group differs. To determine which *specific pairs* of body sites differ:

1. Run PERMANOVA for each pair of body sites (Gut vs Skin, Gut vs Oral, Skin vs Oral)
2. Apply Bonferroni correction to the p-values (multiply by 3)
3. Which pairs are significantly different?

In [ ]:
# Exercise 4: your code here


### Exercise 5: Differential Abundance

Identify which taxa are differentially abundant between body sites:

1. For each taxon, compute its mean relative abundance in each body site
2. Use a Kruskal-Wallis test to identify taxa that differ significantly across sites
3. Apply FDR correction (Benjamini-Hochberg)
4. Visualize the top 10 differentially abundant taxa as a heatmap

This is analogous to differential expression analysis in RNA-seq, but applied to microbiome data.

In [ ]:
# Exercise 5: your code here


### Exercise 6: Mantel Test

The **Mantel test** assesses correlation between two distance matrices. This is useful when asking whether two different diversity metrics give consistent results.

1. Implement a simple Mantel test (correlate the entries of two distance matrices, permute one to get a null distribution)
2. Test whether the Bray-Curtis and simplified UniFrac distance matrices are correlated
3. Do they give the same picture of community relationships?

In [ ]:
# Exercise 6: your code here


---

## Resources

### Essential Reading
- [Callahan et al., 2016](https://doi.org/10.1038/nmeth.3869) -- DADA2 paper
- [Callahan et al., 2017](https://doi.org/10.1038/s41587-019-0209-9) -- "Exact sequence variants should replace OTUs"
- [Bolyen et al., 2019](https://doi.org/10.1038/s41587-019-0209-9) -- QIIME2 paper
- [Knight et al., 2018](https://doi.org/10.1038/s41579-018-0029-9) -- Best practices for microbiome studies
- [Lozupone & Knight, 2005](https://doi.org/10.1128/AEM.71.12.8228-8235.2005) -- UniFrac paper

### Databases
- [SILVA](https://www.arb-silva.de/) -- 16S/18S reference database
- [Greengenes2](https://greengenes2.ucsd.edu/) -- Genome-informed 16S taxonomy
- [GTDB](https://gtdb.ecogenomic.org/) -- Genome Taxonomy Database

### Tools
- [QIIME2](https://qiime2.org/) -- Microbiome analysis platform
- [DADA2](https://benjjneb.github.io/dada2/) -- ASV denoising
- [phyloseq](https://joey711.github.io/phyloseq/) -- R microbiome analysis
- [scikit-bio](http://scikit-bio.org/) -- Python ecology/bioinformatics toolkit
- [MicrobiomeAnalyst](https://www.microbiomeanalyst.ca/) -- Web-based analysis

### Tutorials
- [QIIME2 "Moving Pictures" tutorial](https://docs.qiime2.org/2024.2/tutorials/moving-pictures/)
- [An Introduction to Applied Bioinformatics: Studying Microbial Diversity](http://readiab.org/)

---

[← Previous: RNA-seq Analysis: From Reads to Differential Expression](../../Tier_3_Applied_Bioinformatics/03_RNA_seq_Analysis/01_rna_seq_analysis.ipynb) | [Next: Promoter and Regulatory Sequence Analysis →](../../Tier_3_Applied_Bioinformatics/05_Promoter_and_Regulatory_Analysis/01_promoter_and_regulatory_analysis.ipynb)